# **LangGraph 101: Building Stateful AI Workflows**

In [283]:
import random
import string
from typing import TypedDict

from langgraph.graph import StateGraph, END

class ChainState(TypedDict):
    n: int
    letter: str

initial_state = ChainState(n=1, letter='a')
print(initial_state)

def add(state: ChainState) -> ChainState:
    random_letter = random.choice(string.ascii_lowercase)
    return {**state, "n": state["n"]+1, "letter": random_letter,}

def print_out(state: ChainState) -> ChainState:
    print("Current n:", state["n"], "letter:" ,state["letter"])
    return state

def stop_condition(state: ChainState) -> bool:
    return state["n"] >= 13

{'n': 1, 'letter': 'a'}


In [284]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(ChainState)
workflow.add_node("add", add)
workflow.add_node("print", print_out)
workflow.add_edge("add", "print")
workflow.add_conditional_edges("print", stop_condition,
{
    True: END,
    False: "add"
})

workflow.set_entry_point("add")
app = workflow.compile()
result = app.invoke({"n":1, "letter":""})
# state: {'n':1, "letter":''}


Current n: 2 letter: w
Current n: 3 letter: w
Current n: 4 letter: u
Current n: 5 letter: v
Current n: 6 letter: j
Current n: 7 letter: x
Current n: 8 letter: t
Current n: 9 letter: k
Current n: 10 letter: o
Current n: 11 letter: p
Current n: 12 letter: w
Current n: 13 letter: c


In [285]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx

dotenv.load_dotenv()

llm = ChatOpenAI(
    model = "gpt-4.1-nano",
    api_key = os.getenv("OPEN_AI_KEY"),
)

# watsonx_llm = ChatWatsonx(
#     model_id="ibm/granite-3-2-8b-instruct",
#     url="https://api.ap-south-1.dl.watson-orchestrate.ibm.com/instances/20251212-2008-2496-2060-3f686fc39c52",
#     project_id="your project id associated with the API key",
#     api_key=os.getenv("WATSONX_API_KEY"),
# )

## Components of LangGraph - Authentication workflow

In this example, we are defining a state structure for a **user authentication workflow**. The state, called `AuthState`, is a typed dictionary that holds information about a user's credentials (username and password) and their authentication status (`is_authenticated`) as shown here:

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/dhl478pwbZzrxzmcjJY4xw/Screenshot%202024-12-20%20at%204-19-46%E2%80%AFPM.png" alt="Screenshot" width="300">
</div>


### 1. States

In [286]:
from typing import TypedDict, Optional

class AuthState(TypedDict):
    username: Optional[str]
    password: Optional[str]
    is_authenticated: Optional[bool]
    output: Optional[str]

#### Example Objects and Their States
##### Object 1: Successful Login

In [287]:
auth_state_1: AuthState = {
    "username" : "John",
    "password" : "Password_1",
    "is_authenticated" : True,
    "output" : "Login successful"
}
print(f"auth_state1: {auth_state_1}")

auth_state1: {'username': 'John', 'password': 'Password_1', 'is_authenticated': True, 'output': 'Login successful'}


##### Object 2: Unsuccessful Login

In [288]:
auth_state_2: AuthState = {
    "username" : "",
    "password" : "wrongPassword",
    "is_authenticated" : False,
    "output" : "Authentication failed. Please try again"
}
print(f"auth_state2: {auth_state_2}")

auth_state2: {'username': '', 'password': 'wrongPassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again'}


### 2. Nodes
Nodes are the **core units of action** in LangGraph. Each node represents a specific task or operation that the AI agent needs to perform

#### Input Node

In [289]:
def input_node(state):
    print(state)
    if state.get('username', "") == "":
        state['username'] = input("What is your username?")

    password = input("Enter your password?")
    return {"password": password}

In [290]:
input_node(auth_state_1)

{'username': 'John', 'password': 'Password_1', 'is_authenticated': True, 'output': 'Login successful'}


{'password': 'passw'}

In [291]:
input_node(auth_state_2)

{'username': '', 'password': 'wrongPassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again'}


{'password': 'secure_password'}

#### Validate credential node

In [292]:
def validate_credentials_node(state):
    username = state.get("username", "")
    password = state.get("password", "")

    print("Username: ", username, "Password: ", password)

    if username == "test_user" and password == "secure_password":
        is_authenticated = True
    else:
        is_authenticated = False

    return {"is_authenticated": is_authenticated}

##### Validate node with incorrect format

In [293]:
validate_credentials_node(auth_state_1)

Username:  John Password:  Password_1


{'is_authenticated': False}

Validate node with correct format

In [294]:
auth_state_3: AuthState = {
    "username":"test_user",
    "password":  "secure_password",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_3: {auth_state_3}")

auth_state_3: {'username': 'test_user', 'password': 'secure_password', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


In [295]:
validate_credentials_node(auth_state_3)

Username:  test_user Password:  secure_password


{'is_authenticated': True}

#### Success node

In [296]:
def success_node(state):
    return {"output": "Authentication successful! Welcome!"}

In [297]:
success_node(auth_state_3)

{'output': 'Authentication successful! Welcome!'}

#### Failure node

In [298]:
def failure_node(state):
    return {"output": "Authentication failed. Please try again"}

#### Router node

In [299]:
def router(state):
    if state["is_authenticated"]:
        return "success_node"

    return "failure_node"

### Graph

In [300]:
from langgraph.graph import StateGraph, END
workflow = StateGraph(AuthState)
workflow

#### Add nodes to the graph

In [301]:
workflow.add_node("input_node", input_node)
workflow.add_node("validate_credentials_node", validate_credentials_node)
workflow.add_node("success_node", success_node)
workflow.add_node("failure_node", failure_node)

### 3. Edges

#### Connect the nodes

In [302]:
workflow.add_edge("input_node", "validate_credentials_node")
workflow.add_edge("success_node", END)
workflow.add_edge("failure_node", "input_node")

#### Conditional Edges

In [303]:
workflow.add_conditional_edges ("validate_credentials_node", router, {"success_node": "success_node", "failure_node": "failure_node"})

#### Setting the entry point

In [304]:
workflow.set_entry_point("input_node")

#### Compile the workflow

In [305]:
app = workflow.compile()

### Running the application

In [306]:
inputs = {"username": "test_user", "password": "secure_password"}
result = app.invoke(inputs)
print(result)

{'username': 'test_user', 'password': 'secure_password'}
Username:  test_user Password:  secure_password
{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Authentication successful! Welcome!'}


In [307]:
result['output']

'Authentication successful! Welcome!'

## Components of LangGraph - QA Workflow

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BgP-ruk_KS5H8D7iISsJ6A/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM.png" alt="Screenshot" width="150">
</div>

### QA State

In [308]:
class QAState(TypedDict):
    question: Optional[str]
    context: Optional[str]
    answer: Optional[str]

#### Example state object

In [309]:
qa_state_example = QAState(question="What is the purpose of this guided project?",
                           context="This project focuses on building a chatbot using Python.",
                           answer=None
                           )
for key,value in qa_state_example.items():
    print(f"{key}: {value}")

question: What is the purpose of this guided project?
context: This project focuses on building a chatbot using Python.
answer: None


### Nodes
#### Input Node

In [310]:
def input_validation_node(state):
    question = state.get("question", "").strip()

    if not question:
        return {"valid": False, "error": "Please provide a question"}

    return {"valid": True}

In [311]:
input_validation_node(qa_state_example)

{'valid': True}

#### Context Provider

In [312]:
def context_provider_node(state):
    question = state.get("question", "").lower()

    if "langgraph" in question or "guided project" in question:
        context = (
            "This guided project is about using LangGraph, a Python library to design state-based workflows. "
            "LangGraph simplifies building complex applications by connecting modular nodes with conditional edges."
        )
        return {"context": context}

    return {"context": None}

### LLM integration for QA workflow

In [313]:
def llm_qa_node(state):
    # Extract the question and context from the state
    question = state.get("question", "")
    context = state.get("context", None)

    # Check for missing context and return a fallback response
    if not context:
        return {"answer": "I don't have enough context to answer your question."}

    # Construct the prompt dynamically
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer the question based on the provided context."

    # Use LangChain's ChatOpenAI to get the response
    try:
        response = llm.invoke(prompt)
        return {"answer": response.content.strip()}
    except Exception as e:
        return {"answer": f"An error occurred: {str(e)}"}

### QA workflow graph

In [314]:
qa_workflow = StateGraph(QAState)
qa_workflow.add_node("input_validation_node", input_validation_node)
qa_workflow.add_node("context_provider_node", context_provider_node)
qa_workflow.add_node("llm_qa_node", llm_qa_node)

qa_workflow.set_entry_point("input_validation_node")

qa_workflow.add_edge("input_validation_node", "context_provider_node")
qa_workflow.add_edge("context_provider_node", "llm_qa_node")
qa_workflow.add_edge("llm_qa_node", END)

qa_app = qa_workflow.compile()

In [315]:
qa_app.invoke({"question": "What is the temperature in Bangalore currently?"})

{'question': 'What is the temperature in Bangalore currently?',
 'context': None,
 'answer': "I don't have enough context to answer your question."}

In [316]:
qa_app.invoke({"question": "What is LangGraph?"})

{'question': 'What is LangGraph?',
 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.',
 'answer': 'An error occurred: Sync client is not available. This happens when an async callable was provided for the API key. Use async methods (ainvoke, astream) instead, or provide a string or sync callable for the API key.'}

## Exercises
### Exercise 1 - Define the State type

Here, define the state schema used by the graph. It should keep track of:
- `n`: a counter starting from 1.
- `letter`: a randomly generated lowercase letter at each step.

In [317]:
class CounterState(TypedDict):
    n: int
    letter: str

In [318]:
counter_state_example = CounterState(n=1, letter="a")
print(f"counter_state: {counter_state_example}")

counter_state: {'n': 1, 'letter': 'a'}


### Exercise 2 - Create `add()` node Function

This node should represent the `increment` step such that:
- It adds 1 to the current value of n.
- It randomly selects a lowercase letter and updates the letter field.

In [319]:
def add_node(state):
    random_letter = random.choice(string.ascii_lowercase)
    return {**state, "n": state["n"]+1, "letter": random_letter}

### Exercise 3 - Create `print_out()` node Function

This node should print the current state such that:
- It logs the value of n and the current random letter.
- The state is returned.


In [320]:
def print_out_node(state):
    print("Current n:", state["n"], "Letter:", state["letter"])
    return state

### Exercise 4 - Stop Condition

Create a function that has a termination condition:
- If the counter reaches 13 or more, the workflow should end.
- Otherwise, it should loop back to add node.


In [321]:
def stop_router(state):
    return state["n"] >= 13

### Exercise 5 - Graph Construction

In this exercise, you'll build the LangGraph flow:

- Create a `StateGraph` object using the `ChainState` that you made.
- Add nodes `add` and `print`.
- Add an edge between `add` and `print`
- Add a conditional edge between `print` and `END` based on `stop_condition`.
- Set `add` as entry point of the graph.


In [322]:
counter_workflow = StateGraph(CounterState)
counter_workflow.add_node("add_node", add_node)
counter_workflow.add_node("print_out_node", print_out_node)
counter_workflow.add_node("stop_node", stop_router)

counter_workflow.add_edge("add_node", "print_out_node")
counter_workflow.add_conditional_edges ("print_out_node", stop_router, {
    True: END,
    False: "add_node",
})

counter_workflow.set_entry_point("add_node")
counter_app = counter_workflow.compile()

In [326]:
result = counter_app.invoke({"n": 1, "letter": "a"})

Current n: 2 Letter: o
Current n: 3 Letter: j
Current n: 4 Letter: a
Current n: 5 Letter: c
Current n: 6 Letter: a
Current n: 7 Letter: q
Current n: 8 Letter: q
Current n: 9 Letter: v
Current n: 10 Letter: b
Current n: 11 Letter: q
Current n: 12 Letter: i
Current n: 13 Letter: z
